In [1]:
from glob import glob
import pandas as pd
import os
from shutil import copyfile

In [2]:
old_models_info = pd.read_csv("old_models_info.csv")
xgb_models_info = pd.read_csv("xgb_models_info.csv")

In [3]:
len(old_models_info), len(xgb_models_info)

(768, 864)

In [4]:
old_models_info.head()

,name,f1_score,roc_auc_score,total_actives,predicted_actives,version
0,transmembrane_protease_serine_11d,1.0,1.0,4,4,py35
1,farnesyl_pyrophosphate_synthase,1.0,1.0,36,36,py27
2,c3a_anaphylatoxin_chemotactic_receptor,1.0,1.0,5,5,py35
3,bombesin_receptor_subtype_3,1.0,1.0,24,24,py35
4,fatty_acid_desaturase_1,1.0,1.0,7,7,py27


In [5]:
xgb_models_info.head()

,model_name,f1,roc_auc,accuracy
0,15_hydroxyprostaglandin_dehydrogenase_nad_,0.928571,0.999333,0.996745
1,1_25_dihydroxyvitamin_d_3_24_hydroxylase_mitoc...,1.000000,1.000000,1.000000
2,1_acyl_sn_glycerol_3_phosphate_acyltransferase...,1.000000,1.000000,1.000000
3,2_acylglycerol_o_acyltransferase_2,0.967742,0.999848,0.998465
4,3_beta_hydroxysteroid_dehydrogenase_delta_5_4_...,0.888889,0.994624,0.989691


In [6]:
# Rename the models name column to match with the other dataframe
xgb_models_info = xgb_models_info.rename(index=str, columns={'model_name': 'name'})

In [7]:
common_df = pd.merge(old_models_info, xgb_models_info, how='inner', on='name')
common_df.head()

,name,f1_score,roc_auc_score,total_actives,predicted_actives,version,f1,roc_auc,accuracy
0,transmembrane_protease_serine_11d,1.0,1.0,4,4,py35,0.888889,1.000000,0.996785
1,farnesyl_pyrophosphate_synthase,1.0,1.0,36,36,py27,1.000000,1.000000,1.000000
2,c3a_anaphylatoxin_chemotactic_receptor,1.0,1.0,5,5,py35,1.000000,1.000000,1.000000
3,bombesin_receptor_subtype_3,1.0,1.0,24,24,py35,0.897959,0.999565,0.993679
4,fatty_acid_desaturase_1,1.0,1.0,7,7,py27,0.857143,0.997592,0.989189


In [8]:
good_old_models = common_df[common_df.f1_score > common_df.f1]
good_old_models.to_csv("good_old_models.csv", index=None)

In [9]:
old_models = []
xgb_models =[]

In [11]:
# copy good old models
count = 0
for _model in good_old_models.name:
    old_models.append(_model)
    if not os.path.isfile('bin/old_models/' + _model): print("{} NOT FOUND".format(_model))
    copyfile('bin/old_models/' + _model, 'models/' + _model)

In [12]:
# copy xgb models
count = 0
for _model in xgb_models_info[~xgb_models_info.name.isin(good_old_models.name)]['name']:
    if not os.path.isfile('bin/xgb_models/' + _model): 
        print("{} NOT FOUND".format(_model))
        continue
    xgb_models.append(_model)
    copyfile('bin/xgb_models/' + _model, 'models/' + _model)

In [31]:
# list the python 2.7 and python 3.5 models (crap!!!)
py35_models = xgb_models_info[~xgb_models_info.name.isin(good_old_models.name)]['name'].tolist()
py35_models.extend(good_old_models[good_old_models.version=='py35']['name'].tolist())

py27_models = good_old_models[good_old_models.version=='py27']['name'].tolist()

In [32]:
with open('py35_models.txt', 'w') as f:
    f.write(','.join(py35_models))

In [33]:
with open('py27_models.txt', 'w') as f:
    f.write(','.join(py27_models))

In [34]:
print("py35 models: {} and py27 models: {}".format(len(py35_models), len(py27_models)))

py35 models: 753 and py27 models: 111


In [35]:
len(xgb_models) + len(old_models)

865

In [14]:
# convert pickled xgb models to joblib models

In [17]:
from sklearn.externals import joblib
import pickle

In [22]:
for _model in xgb_models:
    model = pickle.load(open('models/' + _model, 'rb'))
    with open('xgb_joblib/'+_model, 'wb') as f:
        joblib.dump(model, f)
    #break

/home/mhassan/anaconda3/envs/dl/lib/python3.5/site-packages/sklearn/base.py:311: UserWarning: Trying to unpickle estimator LabelEncoder from version 0.20.2 when using version 0.19.1. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)


In [39]:
# test
py35_models = open('py35_models.txt', 'r').readlines()[0].split(',')

In [42]:
for _model in py35_models:
    joblib.load('models/' + _model)

/home/mhassan/anaconda3/envs/dl/lib/python3.5/site-packages/sklearn/base.py:311: UserWarning: Trying to unpickle estimator DecisionTreeClassifier from version 0.18.1 when using version 0.19.1. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)
/home/mhassan/anaconda3/envs/dl/lib/python3.5/site-packages/sklearn/base.py:311: UserWarning: Trying to unpickle estimator RandomForestClassifier from version 0.18.1 when using version 0.19.1. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)
/home/mhassan/anaconda3/envs/dl/lib/python3.5/site-packages/sklearn/base.py:311: UserWarning: Trying to unpickle estimator SVC from version 0.18.1 when using version 0.19.1. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)


In [43]:
# test to predict
import sys
sys.path.append("../ddt/")
from utility import FeatureGenerator

In [44]:
ft = FeatureGenerator()
ft.load_sdf("../ddt/samples/AAAAML.xaa.sdf")

In [56]:
def get_models(version='35'):
    model_names = open('py' + version + '_models.txt', 'r').readlines()[0].split(',')
    for model in model_names:
        yield model, joblib.load('models/' + model)

In [66]:
features = ft.extract_tpatf()
for model_name, model in get_models():
    if type(model).__name__ == "SVC": 
        predict = model.predict(features)
        if predict.any():
            print(model_name, predict)
    else:
        predict = model.predict_proba(features)
        conf_mask = predict[:, 1]>0.9
        print(predict[conf_mask])
    
    #break

[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[[0.03089797 0.969102  ]
 [0.02369303 0.976307  ]]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[[0.00878769 0.9912123 ]
 [0.00878769 0.9912123 ]]
[[0.03521597 0.964784  ]
 [0.04526317 0.9547368 ]]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[[9.2879772e-02 9.0712023e-01]
 [9.4372034e-04 9.9905628e-01]
 [9.4372034e-04 9.9905628e-01]
 [9.4372034e-04 9.9905628e-01]
 [2.0657778e-03 9.9793422e-01]
 [8.5902214e-04 9.9914098e-01]
 [4.4864416e-04 9.9955136e-01]
 [9.4372034e-04 9.9905628e-01]]
[]
[]
[]
[]
[]
[]
[[0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]
 [0.00451684 0.99548316]]
[]
[]
[]
[]
[]
[[0.04274112 0.9572589 ]
 [0.04239851 0.9576015 ]
 [0.04344296 0.95655704]
 [0.07467812 0.92532

[]
[]
[]
[]
[]
[]
[]
[[0.00223351 0.9977665 ]
 [0.00331146 0.99668854]
 [0.00510448 0.9948955 ]
 [0.00510448 0.9948955 ]]
[]
[[0.02312654 0.97687346]
 [0.08421278 0.9157872 ]
 [0.08421278 0.9157872 ]
 [0.08421278 0.9157872 ]
 [0.08421278 0.9157872 ]]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[[0.0049547  0.9950453 ]
 [0.00638342 0.9936166 ]]
[]
[]


/home/mhassan/anaconda3/envs/dl/lib/python3.5/site-packages/sklearn/base.py:311: UserWarning: Trying to unpickle estimator DecisionTreeClassifier from version 0.18.1 when using version 0.19.1. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)
/home/mhassan/anaconda3/envs/dl/lib/python3.5/site-packages/sklearn/base.py:311: UserWarning: Trying to unpickle estimator RandomForestClassifier from version 0.18.1 when using version 0.19.1. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)


[]
[]


/home/mhassan/anaconda3/envs/dl/lib/python3.5/site-packages/sklearn/base.py:311: UserWarning: Trying to unpickle estimator SVC from version 0.18.1 when using version 0.19.1. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)


[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
lysine_specific_demethylase_5c [1. 0. 0. 0. 1. 1. 1. 0. 0. 0. 1. 0. 1. 0. 0.]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
